# QDGrasp Phase 3 CUDA Gate & Multi-Angle (4-View) Grasp Video Suite

Verification gate and 4-camera video generation running on NVIDIA Tesla T4 GPU on Kaggle.

### Contents:
1. **Phase 1**: PyTorch CUDA FP32 train, AMP (`16-mixed`), bit-exact optimizer resume.
2. **Phase 2**: Forward Kinematics (FK) parity between CPU and CUDA for LEAP Hand, Wonik Allegro, Shadow Hand.
3. **Phase 3.2**: Underactuated Control, Rank-20 Moment Matrix FD parity, controllable-space projection.
4. **Phase 3.2.1**: 100-Case Stress Matrix & Multi-Embodiment Rollout Gate.
5. **Phase 3.3**: Scene Grasping Dataset Schema, Sharding, and Deterministic Release Parity.
6. **Phase 3.4**: Multi-Angle (4-View) Grasp Rollout Video Suite across 3 robot embodiments (LEAP Hand, Wonik Allegro, Shadow Hand) and procedural shapes.


In [ ]:
import os
import subprocess
import sys

assert sys.version_info >= (3, 11), f'Python >=3.11 required, got {sys.version}'

# 1. Install dependencies and qdgrasp from branch
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade',
    'lightning>=2.6.0', 'mujoco>=3.3.0', 'numpy>=2.0.0',
    'scipy>=1.14.0', 'trimesh>=4.0.0', 'safetensors>=0.5.0',
    'pydantic>=2.10.0', 'PyYAML>=6.0.0', 'einops>=0.8.0',
    'rich>=13.0.0', 'typer>=0.16.0', 'imageio[ffmpeg]', 'Pillow', 'pytest',
], check=True)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--quiet', '--no-deps', '--force-reinstall',
    'git+https://github.com/ninicom/qdgrasp.git@feature/phase3-data-layer',
], check=True)

# 2. Clone repository scripts and mujoco-menagerie for robot assets into /tmp
assets_dir = '/tmp/robot-assets/mujoco-menagerie'
if not os.path.exists(assets_dir):
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/google-deepmind/mujoco_menagerie.git', assets_dir,
    ], check=True)

repo_dir = '/tmp/qdgrasp_repo'
if not os.path.exists(repo_dir):
    subprocess.run([
        'git', 'clone', '--depth', '1', '-b', 'feature/phase3-data-layer',
        'https://github.com/ninicom/qdgrasp.git', repo_dir,
    ], check=True)

print('QDGrasp & dependencies installed successfully on Python', sys.version.split()[0])


In [ ]:
# Phase 1, 2, 3 Verification Gates & Audits
import os
import subprocess
import sys

runner_code = '''
import json
import os
import torch
from pathlib import Path
from datetime import datetime, timezone

os.environ["QDGRASP_ROBOT_ASSETS_ROOT"] = "/tmp/robot-assets"

import qdgrasp
from qdgrasp import environment_info, require_cuda
from qdgrasp.api import QDGrasp
from qdgrasp.config.loader import load_robot_config
from qdgrasp.engine.callbacks import LossHistory
from qdgrasp.robot.spec import RobotSpec

print("==================================================")
print("      QDGrasp Kaggle GPU Gate & Benchmark         ")
print("==================================================")

assert torch.cuda.is_available(), "Kaggle GPU gate requires NVIDIA GPU"
require_cuda(expected_runtime=torch.version.cuda)
print(f"CUDA Available  : {torch.cuda.is_available()}")
print(f"GPU Device      : {torch.cuda.get_device_name(0)}")
print(f"CUDA Capability : {torch.cuda.get_device_capability(0)}")
print(f"PyTorch CUDA    : {torch.version.cuda}")
print(f"QDGrasp Version : {qdgrasp.__version__}")
print("==================================================\\n")

# Phase 1: CUDA Training & AMP
print("--> Running Phase 1 CUDA FP32 & AMP Training...")
fp32_hist = LossHistory()
res_fp32 = QDGrasp().train("dummy-tiny.yaml", device="cuda:0", max_steps=8, batch_size=4, val_interval=4, run_name="cuda-fp32", callbacks=[fp32_hist])
print(f"Phase 1 FP32 Loss: {res_fp32.final_loss:.4f} (steps={res_fp32.global_step})")

amp_hist = LossHistory()
res_amp = QDGrasp().train("dummy-tiny.yaml", device="cuda:0", amp=True, max_steps=8, batch_size=4, run_name="cuda-amp", callbacks=[amp_hist])
print(f"Phase 1 AMP Loss : {res_amp.final_loss:.4f} (precision={res_amp.runtime['effective']['precision']})")
assert res_amp.runtime["effective"]["precision"] == "16-mixed", "AMP precision mismatch"
print("Phase 1 CUDA Gate: PASS\\n")

# Phase 2: Forward Kinematics CUDA Parity
print("--> Running Phase 2 Forward Kinematics Parity...")
for preset_name in ["leap_hand.yaml", "wonik_allegro.yaml", "shadow_hand.yaml"]:
    cfg = load_robot_config(preset_name)
    spec = RobotSpec.from_config(cfg, sample_anchors=False)
    J = len(spec.actuated_joint_names)
    palm_pos_cpu = torch.randn(16, 3, dtype=torch.float32)
    palm_rot_cpu = torch.eye(3, dtype=torch.float32).unsqueeze(0).expand(16, 3, 3).clone()
    joints_cpu = torch.randn(16, J, dtype=torch.float32) * 0.2
    
    palm_pos_cuda = palm_pos_cpu.cuda()
    palm_rot_cuda = palm_rot_cpu.cuda()
    joints_cuda = joints_cpu.cuda()
    
    t_cpu = spec.forward_kinematics(palm_pos_cpu, palm_rot_cpu, joints_cpu)
    t_cuda = spec.forward_kinematics(palm_pos_cuda, palm_rot_cuda, joints_cuda)
    
    max_diff = 0.0
    for link_name, mat_cpu in t_cpu.items():
        mat_cuda = t_cuda[link_name].cpu()
        diff = float((mat_cpu - mat_cuda).abs().max().item())
        if diff > max_diff:
            max_diff = diff
    assert max_diff < 1e-5, f"FK parity failed on {preset_name}: max_diff={max_diff}"
    print(f"Phase 2 FK parity on {preset_name}: PASS (max_diff={max_diff:.2e})")
print("Phase 2 CUDA Gate: PASS\\n")

# Phase 3: GPU Training Benchmark
print("--> Running Phase 3 GPU Training Benchmark...")
grasper_p3 = QDGrasp("qdgrasp-dummy-n.yaml", robot="leap_hand.yaml", seed=42)
res_p3 = grasper_p3.train(
    "dummy-tiny.yaml",
    device="cuda",
    max_steps=50,
    batch_size=16,
    learning_rate=1e-3,
    run_name="kaggle_p3_benchmark",
)
print("Phase 3 GPU Training: PASS")
print("Final Metrics:", res_p3.metrics)

evidence = {
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "device": torch.cuda.get_device_name(0),
    "cuda_version": torch.version.cuda,
    "phase1_fp32_loss": res_fp32.final_loss,
    "phase1_amp_loss": res_amp.final_loss,
    "phase3_metrics": res_p3.metrics,
    "status": "PASS",
}
Path("phase3_cuda_evidence.json").write_text(json.dumps(evidence, indent=2))
print("\\nSaved evidence to phase3_cuda_evidence.json")
'''

env = dict(
    os.environ,
    CUBLAS_WORKSPACE_CONFIG=':4096:8',
    QDGRASP_ROBOT_ASSETS_ROOT='/tmp/robot-assets',
    PYTHONHASHSEED='0',
    OMP_NUM_THREADS='1',
    MKL_NUM_THREADS='1',
    OPENBLAS_NUM_THREADS='1',
)
subprocess.run([sys.executable, '-c', runner_code], env=env, check=True)

# Run Phase 3 Audits directly from cloned repo
print('\n--> Running Phase 3.2, 3.2.1, and 3.3 Verification Audits...')
subprocess.run([sys.executable, '/tmp/qdgrasp_repo/scripts/check_phase3_2.py'], env=env, check=True)
subprocess.run([sys.executable, '/tmp/qdgrasp_repo/scripts/check_phase3_2_1.py'], env=env, check=True)
subprocess.run([sys.executable, '/tmp/qdgrasp_repo/scripts/check_phase3_3.py'], env=env, check=True)
print('\n==================================================')
print('ALL PHASE 3 GATES & AUDITS PASSED (100%)!')
print('==================================================')


## Phase 3.4: Multi-Angle (4-View) Grasp Rollout Video Suite

Renders 4 synchronized virtual camera perspectives (Isometric 45°, Front 0°, Side 90°, Top-Down -85°) into 2x2 grid `.mp4` videos for test grasp rollouts across **LEAP Hand**, **Wonik Allegro**, and **Shadow Hand** on procedural shapes.
Videos are organized into `videos/pass/` and `videos/pal/`.

In [ ]:
# 4. Multi-Angle (4-View) Video Generation
import os
import sys
import json
import base64
import subprocess
from pathlib import Path
from IPython.display import HTML, display

video_runner_code = '''
import os
import json
from pathlib import Path

os.environ["QDGRASP_ROBOT_ASSETS_ROOT"] = "/tmp/robot-assets"
os.environ["MUJOCO_GL"] = "egl"

import sys
sys.path.insert(0, "/tmp/qdgrasp_repo")
from scripts.render_4view_rollout import run_kaggle_video_suite

output_dir = Path("/kaggle/working/videos")
results = run_kaggle_video_suite(output_dir=output_dir, robot_assets_root="/tmp/robot-assets")
Path("/kaggle/working/video_manifest.json").write_text(json.dumps(results, indent=2))
print("Video suite completed! Manifest saved to /kaggle/working/video_manifest.json")
'''

# Run rendering in headless subprocess with EGL / OS facilities
env = dict(os.environ, MUJOCO_GL='egl', QDGRASP_ROBOT_ASSETS_ROOT='/tmp/robot-assets')
subprocess.run([sys.executable, '-c', video_runner_code], env=env, check=True)

# Load results and render inline HTML video players
manifest_p = Path('/kaggle/working/video_manifest.json')
if manifest_p.exists():
    results = json.loads(manifest_p.read_text())
    print(f'\n================================================================================')
    print(f'🎬 Multi-Angle (4-View) Grasp Videos Generated ({len(results)} Scenarios)')
    print(f'================================================================================\n')
    
    pass_results = [r for r in results if r.get('category') == 'pass']
    pal_results = [r for r in results if r.get('category') in ('pal', 'fail')]
    
    print(f'=== [PASS CATEGORY: {len(pass_results)} Videos in videos/pass/] ===')
    for res in pass_results:
        vid_p = Path(res['video_path'])
        if vid_p.exists() and res['status'] == 'SUCCESS':
            b64_data = base64.b64encode(vid_p.read_bytes()).decode('utf-8')
            card_html = f'''
            <div style="margin-bottom: 28px; padding: 16px; background: #142718; border: 1px solid #22c55e; border-radius: 10px; color: #f4f4f5; font-family: sans-serif;">
                <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 10px;">
                    <h3 style="margin: 0; font-size: 1.15rem; color: #86efac;">&#x2705; [PASS] {res['robot']} &times; {res['object']} ({res['scenario']})</h3>
                    <span style="background: #166534; color: #bbf7d0; font-weight: bold; padding: 4px 10px; border-radius: 6px; font-size: 0.85rem;">&#x2714; CERTIFIED LIFT PASS</span>
                </div>
                <p style="margin: 4px 0 12px 0; color: #cbd5e1; font-size: 0.9rem;">
                    Dir: <code style="color: #86efac; background: #052e16; padding: 2px 6px; border-radius: 4px;">videos/pass/{vid_p.name}</code> | Size: <b>{res['file_size']:,} bytes</b> | 4-View Layout: [Isometric 45&deg; | Front 0&deg; | Side 90&deg; | Top -85&deg;]
                </p>
                <video width="960" height="720" controls autoplay loop muted style="border-radius: 8px; border: 1px solid #16a34a; width: 100%; max-width: 960px; display: block;">
                    <source src="data:video/mp4;base64,{b64_data}" type="video/mp4">
                    Your browser does not support HTML5 video.
                </video>
            </div>
            '''
            display(HTML(card_html))
    
    print(f'\n=== [PAL / STRESS CATEGORY: {len(pal_results)} Videos in videos/pal/] ===')
    for res in pal_results:
        vid_p = Path(res['video_path'])
        if vid_p.exists() and res['status'] == 'SUCCESS':
            b64_data = base64.b64encode(vid_p.read_bytes()).decode('utf-8')
            card_html = f'''
            <div style="margin-bottom: 28px; padding: 16px; background: #2a1515; border: 1px solid #ef4444; border-radius: 10px; color: #f4f4f5; font-family: sans-serif;">
                <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 10px;">
                    <h3 style="margin: 0; font-size: 1.15rem; color: #fca5a5;">&#x274C; [PAL] {res['robot']} &times; {res['object']} ({res['scenario']})</h3>
                    <span style="background: #991b1b; color: #fecaca; font-weight: bold; padding: 4px 10px; border-radius: 6px; font-size: 0.85rem;">&#x2718; EXPECTED SLIP / REACH LIMIT</span>
                </div>
                <p style="margin: 4px 0 12px 0; color: #cbd5e1; font-size: 0.9rem;">
                    Dir: <code style="color: #fca5a5; background: #450a0a; padding: 2px 6px; border-radius: 4px;">videos/pal/{vid_p.name}</code> | Size: <b>{res['file_size']:,} bytes</b> | 4-View Layout: [Isometric 45&deg; | Front 0&deg; | Side 90&deg; | Top -85&deg;]
                </p>
                <video width="960" height="720" controls autoplay loop muted style="border-radius: 8px; border: 1px solid #dc2626; width: 100%; max-width: 960px; display: block;">
                    <source src="data:video/mp4;base64,{b64_data}" type="video/mp4">
                    Your browser does not support HTML5 video.
                </video>
            </div>
            '''
            display(HTML(card_html))
